<a href="https://colab.research.google.com/github/ruchit0002/Autoreq-agentic-data-integrity/blob/main/AutoReg_Agentic_AI_Data_Integrity.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
from sklearn.ensemble import IsolationForest

In [2]:
from google.colab import files
uploaded = files.upload()

Saving Loan_approval_data.csv to Loan_approval_data.csv


In [3]:
file = list(uploaded.keys())[0]

df = pd.read_csv(file)
df.columns = df.columns.str.strip()

df.head()

,customer_id,age,occupation_status,years_employed,annual_income,credit_score,credit_history_years,savings_assets,current_debt,defaults_on_file,delinquencies_last_2yrs,derogatory_marks,product_type,loan_intent,loan_amount,interest_rate,debt_to_income_ratio,loan_to_income_ratio,payment_to_income_ratio,loan_status
0,CUST100000,40,Employed,17.2,25579,692,5.3,895,10820,0,0,0,Credit Card,Business,600,17.02,0.423,0.023,0.008,1
1,CUST100001,33,Employed,7.3,43087,627,3.5,169,16550,0,1,0,Personal Loan,Home Improvement,53300,14.10,0.384,1.237,0.412,0
2,CUST100002,42,Student,1.1,20840,689,8.4,17,7852,0,0,0,Credit Card,Debt Consolidation,2100,18.33,0.377,0.101,0.034,1
3,CUST100003,53,Student,0.5,29147,692,9.8,1480,11603,0,1,0,Credit Card,Business,2900,18.74,0.398,0.099,0.033,1
4,CUST100004,32,Employed,12.5,63657,630,7.2,209,12424,0,0,0,Personal Loan,Education,99600,13.92,0.195,1.565,0.522,1


In [4]:
num_cols = df.select_dtypes(include=["int64","float64"]).columns

if len(num_cols) == 0:
    print("No numerical columns found.")

In [5]:
X = df[num_cols]

model = IsolationForest(contamination=0.05, random_state=42)
model.fit(X)

IsolationForest(contamination=0.05, random_state=42)

In [6]:
pred = model.predict(X)
df["anomaly"] = pred

anomalies = df[df["anomaly"] == -1].copy()

In [7]:
print("=== Detected Anomalies (Table) ===\n")
anomalies

=== Detected Anomalies (Table) ===



,customer_id,age,occupation_status,years_employed,annual_income,credit_score,credit_history_years,savings_assets,current_debt,defaults_on_file,...,derogatory_marks,product_type,loan_intent,loan_amount,interest_rate,debt_to_income_ratio,loan_to_income_ratio,payment_to_income_ratio,loan_status,anomaly
8,CUST100008,29,Employed,5.9,28416,569,2.6,1334,22668,1,...,0,Credit Card,Education,33800,22.72,0.798,1.189,0.396,0,-1
27,CUST100027,39,Employed,0.0,43085,579,3.5,206,359,0,...,1,Personal Loan,Medical,82800,15.68,0.008,1.922,0.641,0,-1
54,CUST100054,47,Self-Employed,4.2,172243,820,3.3,85551,10893,0,...,0,Line of Credit,Business,41700,6.31,0.063,0.242,0.081,1,-1
65,CUST100065,51,Employed,18.7,34543,592,30.0,52072,6288,0,...,1,Credit Card,Business,23000,22.49,0.182,0.666,0.222,0,-1
106,CUST100106,57,Self-Employed,26.6,116531,647,27.2,241834,26197,0,...,0,Credit Card,Business,45700,19.32,0.225,0.392,0.131,1,-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49929,CUST149929,38,Employed,16.1,39629,518,18.4,44963,4462,0,...,1,Credit Card,Home Improvement,5200,22.63,0.113,0.131,0.044,1,-1
49933,CUST149933,54,Student,0.9,31505,552,3.5,4069,2922,1,...,0,Credit Card,Medical,3200,21.22,0.093,0.102,0.034,0,-1
49966,CUST149966,42,Employed,15.8,99696,662,13.2,39516,62417,0,...,0,Personal Loan,Education,6900,11.63,0.626,0.069,0.023,0,-1
49969,CUST149969,60,Employed,32.4,76612,737,27.0,5045,37790,0,...,0,Credit Card,Personal,70000,16.59,0.493,0.914,0.305,1,-1


In [8]:
summary_rows = []

for idx, row in anomalies.iterrows():
    for col in num_cols:
        value = row[col]
        mean = df[col].mean()
        std = df[col].std()

        if abs(value - mean) > 3 * std:
            summary_rows.append({
                "Row": idx,
                "Column": col,
                "Value": value,
                "Mean": round(mean,2),
                "StdDev": round(std,2)
            })

summary = pd.DataFrame(summary_rows)

In [9]:
from IPython.display import display

print("=== Anomaly Explanation Table ===\n")

if len(summary) > 0:
    display(summary)
else:
    print("No abnormal columns detected.")

=== Anomaly Explanation Table ===



,Row,Column,Value,Mean,StdDev
0,8,defaults_on_file,1.000,0.05,0.22
1,8,debt_to_income_ratio,0.798,0.29,0.16
2,27,delinquencies_last_2yrs,4.000,0.55,0.85
3,54,annual_income,172243.000,50062.89,32630.50
4,54,savings_assets,85551.000,3595.62,13232.40
...,...,...,...,...,...
3000,49927,current_debt,59826.000,14290.44,13243.76
3001,49929,savings_assets,44963.000,3595.62,13232.40
3002,49933,defaults_on_file,1.000,0.05,0.22
3003,49966,current_debt,62417.000,14290.44,13243.76


In [10]:
print("Total anomalies detected:", len(anomalies))

Total anomalies detected: 2500
